### ⚙️ Initial Setup

In [0]:
import pyspark.sql.functions as sf
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.window import Window

# style like R ggplot
plt.style.use("ggplot")

In [0]:
# base volume path
BASE_DIR = "/Volumes/workspace/default/home-credit-default-risk"

In [0]:
# reading data
pmt = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/installments_payments.csv")
)

pos = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/POS_CASH_balance.csv")
)

cc = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/credit_card_balance.csv")
)

### 1. 💰 Installments Payments

The `Installments Payments` dataset tracks the detailed, transaction-level repayment history for past loans disbursed by Home Credit. Every row represents a single scheduled payment installment, recording both what was contracted (the due date and required installment amount) alongside what actually occurred (the date the borrower paid and the exact dollar amount deposited).

In [0]:
print(f"({pmt.count()}, {len(pmt.columns)})")

In [0]:
pmt.show(5)

#### NUM_INSTALMENT_VERSION

This column tracks how many times a loan's payment schedule was modified or recalculated over its lifetime. Think of it as a counter for schedule revisions and payment plan changes.

* **`1` (Original Plan):** Standard payments made under the original agreement, where neither the due dates nor the required installment amounts were ever altered.
* **`> 1` (2, 3, 4... Revised Plan):** The schedule was recalculated or renegotiated. Every adjustment increments this counter—higher numbers reflect repeated loan restructuring, frequent partial/micro-payments, or rolled-over late fees.
* **`0` (Flexible / Revolving Plan):** Used for revolving credit lines or credit cards where there is no fixed repayment schedule, only a dynamic monthly minimum.

In [0]:
pmt.groupBy("NUM_INSTALMENT_VERSION").count().orderBy("count", ascending=False).show(4)

In [0]:
instalment_df = (
    pmt
    .groupBy("NUM_INSTALMENT_VERSION")
    .count()
    .orderBy("count", ascending=False)
    .limit(4)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 8))

wedges, _, _ = ax.pie(
    instalment_df["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    instalment_df["NUM_INSTALMENT_VERSION"].astype(str),
    title="Installment Version",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "Top 4 Installment Versions",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

---

#### NUM_INSTALMENT_NUMBER

This column tracks the sequence number of each installment within a specific loan contract. It acts as a step counter for your payments from the beginning to the end of a loan term.

* **`1` (First Payment):** The very first scheduled payment due on the contract.
* **`2, 3, 4...` (Progressive Payments):** The 2nd, 3rd, or 4th payment in the ongoing installment schedule.
* **`N` (Final Payment):** The highest installment number recorded for a closed contract represents the total number of payments required to finish that loan.

#### Maximum Number of Installments for Each Customer
This feature captures the highest installment sequence number an applicant has ever reached across all their past loans combined (`max(NUM_INSTALMENT_NUMBER)` grouped by `SK_ID_CURR`).

In [0]:
pmt.groupBy("SK_ID_CURR").agg(sf.max("NUM_INSTALMENT_NUMBER").alias("MAX_NUM_INSTALMENT_NUMBER")).show(10)

In [0]:
# let's see the kde plot of this
(
    pmt
    .groupBy("SK_ID_CURR")
    .agg(
        sf.max("NUM_INSTALMENT_NUMBER").alias("MAX_NUM_INSTALMENT_NUMBER")
    )
    .select("MAX_NUM_INSTALMENT_NUMBER")
).plot.kde(0.8, title="KDE Plot for MAX_NUM_INSTALMENT_NUMBER Per SK_ID_CURR")

#### Distribution Analysis: MAX_NUM_INSTALMENT_NUMBER Per SK_ID_CURR

* **Primary Peak at 12 Months:** The vast majority of applicants have a maximum installment count centered around **12 payments**, making 1-year loans the most common maximum tenure.
* **Contractual Spikes (6, 18, 24, 36):** The distribution shows distinct peaks at standard financial milestones (6, 12, 18, 24, and 36 months), reflecting standard retail and cash loan terms.
* **Secondary Bump at ~100:** A small, isolated peak appears around 100 installments, capturing long-term revolving credit lines or extended repayment histories (~8+ years of activity).

In [0]:
# let's see the box plot
(
    pmt
    .groupBy("SK_ID_CURR")
    .agg(
        sf.max("NUM_INSTALMENT_NUMBER").alias("MAX_NUM_INSTALMENT_NUMBER")
    )
    .select("MAX_NUM_INSTALMENT_NUMBER")
).plot.box()

*As seen in the KDE plot, the secondary peak suggests the presence of outliers.*

---

#### DAYS_INSTALMENT

This column records the contractual due date for each payment, measured in days relative to the current application date. Because all records belong to past loans, values are negative (e.g., `-30` means due 30 days ago, `-365` means 1 year ago).

Beyond marking individual due dates, this column establishes the full timeline of a customer's history. Tracking its minimum and maximum values reveals how long a borrower has been with Home Credit and how recently they were actively paying.

In [0]:
# histplot
pmt.select("DAYS_INSTALMENT").plot.hist(title="Histogram Plot of DAYS_INSTALMENT")

In [0]:
# box plot
pmt.select("DAYS_INSTALMENT").plot.box()

In [0]:
(
    pmt
    .groupBy("SK_ID_CURR")
    .agg(
        sf.max("DAYS_INSTALMENT").alias("MAX_DAYS_INSTALMENT")
    )
    .select("MAX_DAYS_INSTALMENT")
).plot.kde(0.8, title="KDE Plot for MAX_DAYS_INSTALMENT Per SK_ID_CURR")

In [0]:
(
    pmt
    .groupBy("SK_ID_CURR")
    .agg(
        sf.min("DAYS_INSTALMENT").alias("MIN_DAYS_INSTALMENT")
    )
    .select("MIN_DAYS_INSTALMENT")
).plot.kde(0.8, title="KDE Plot for MIN_DAYS_INSTALMENT Per SK_ID_CURR")

---

#### `DAYS_ENTRY_PAYMENT` (Actual Payment Date)

A relative time index indicating exactly when the borrower *actually* made a payment on a historical loan installment. 

Like other time features in this dataset, it is measured in days relative to the current loan application date, represented as negative integers:
* **`0`**: The payment was made on the exact day of the current loan application.
* **`-30`**: The payment was made roughly one month prior to the current application.
* **`-365`**: The payment was made exactly one year prior to the current application.

In [0]:
pmt.select("DAYS_ENTRY_PAYMENT").show(5)

In [0]:
# hist plot
pmt.select("DAYS_ENTRY_PAYMENT").plot.hist(title="Histogram Plot of DAYS_ENTRY_PAYMENT")

In [0]:
# box plot
pmt.select("DAYS_ENTRY_PAYMENT").plot.box()

In [0]:
# summary stats
pmt.select("DAYS_ENTRY_PAYMENT").summary().show()

In [0]:
pmt.select("DAYS_ENTRY_PAYMENT").orderBy(
    sf.col("DAYS_ENTRY_PAYMENT").asc_nulls_last() # push nulls at last
).show(10)

---

#### AMT_INSTALMENT and AMT_PAYMENT

These two columns capture the financial side of each payment by comparing what was billed against what was actually received:

* **`AMT_INSTALMENT`:** The required installment amount scheduled by the bank for that due date.
* **`AMT_PAYMENT`:** The actual amount the customer paid towards that installment.

Comparing the two (`AMT_PAYMENT - AMT_INSTALMENT`) creates a direct measure of repayment behavior. A negative result highlights underpayments or partial payments-a strong indicator of financial tightness or distress-while positive or equal results show full payments or early payoffs.



In [0]:
pmt.select("AMT_PAYMENT", "AMT_INSTALMENT").show(10)

In [0]:
df_payments = (
    pmt
    .withColumn(
        "PAYMENT_CASE",
        sf.when(sf.col("AMT_PAYMENT") < sf.col("AMT_INSTALMENT"), "UNDERPAID")
        .when(sf.col("AMT_PAYMENT") > sf.col("AMT_INSTALMENT"), "OVERPAID")
        .when(sf.col("AMT_PAYMENT") == sf.col("AMT_INSTALMENT"), "EXACT")
        .otherwise("UNKNOWN")
    )
    .select("PAYMENT_CASE")
)

*Note: UNKNOWN means either AMT_PAYMENT is NULL or AMT_INSTALMENT is NULL*

In [0]:
df_payments.show(5)

In [0]:
df_payments.groupBy("PAYMENT_CASE").count().show()

In [0]:
payment_case_df = df_payments.groupBy("PAYMENT_CASE").count().toPandas()

# Sort for better visualization
payment_case_df = payment_case_df.sort_values("count", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))

sns.barplot(
    data=payment_case_df,
    x="PAYMENT_CASE",
    y="count",
    palette="viridis",
    edgecolor="black",
    linewidth=1.2,
    hue="PAYMENT_CASE",
    ax=ax
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=3,
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "Payment Behavior Distribution",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Payment Case", fontsize=12)
ax.set_ylabel("Number of Installments", fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

#### Summary Stats and Outliers Analysis

In [0]:
pmt.select("AMT_PAYMENT", "AMT_INSTALMENT").describe().show()

In [0]:
quantiles = pmt.approxQuantile(
    "AMT_PAYMENT",
    [0.0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.98, 1.0],
    0.01
)

print(f"Min: {quantiles[0]:,.2f}")
print(f"25%: {quantiles[1]:,.2f}")
print(f"50%: {quantiles[2]:,.2f}")
print(f"75%: {quantiles[3]:,.2f}")
print(f"90%: {quantiles[4]:,.2f}")
print(f"95%: {quantiles[5]:,.2f}")
print(f"Max: {quantiles[7]:,.2f}")

In [0]:
quantiles = pmt.approxQuantile(
    "AMT_INSTALMENT",
    [0.0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.98, 1.0],
    0.01
)

print(f"Min: {quantiles[0]:,.2f}")
print(f"25%: {quantiles[1]:,.2f}")
print(f"50%: {quantiles[2]:,.2f}")
print(f"75%: {quantiles[3]:,.2f}")
print(f"90%: {quantiles[4]:,.2f}")
print(f"95%: {quantiles[5]:,.2f}")
print(f"Max: {quantiles[7]:,.2f}")

##### What is Significance of Zero??

In [0]:
pmt.filter(sf.col("AMT_PAYMENT") == 0).count()

In [0]:
pmt.filter(sf.col("AMT_INSTALMENT") == 0).count()

In [0]:
# case when AMT_PAYMENT = AMT_INSTALMENT = 0
pmt.filter((sf.col("AMT_PAYMENT") == 0) & (sf.col("AMT_INSTALMENT") == 0)).count()

In [0]:
# case when AMT_PAYMENT = 0 and AMT_INSTALMENT > 0
pmt.filter((sf.col("AMT_PAYMENT") == 0) & (sf.col("AMT_INSTALMENT") > 0)).count()

#### Analysis of Zero-Payment Events (`AMT_PAYMENT == 0`)

An inspection of zero-value actual payments reveals two distinct scenarios in the dataset:

* **Complete Missed Payments (1,438 rows):** Instances where `AMT_INSTALMENT > 0` but `AMT_PAYMENT == 0`. These represent total installment defaults where a bill was issued but no money was collected.
* **System Artifacts (2 rows):** Instances where both `AMT_INSTALMENT` and `AMT_PAYMENT` equal `0`. These are rare zero-balance administrative adjustments with negligible statistical impact.

#### Null Values Analysis

In [0]:
# null values
null_counts = pmt.select([
    sf.count(sf.when(sf.col(c).isNull(), c)).alias(c)
    for c in pmt.columns
])
null_counts.show(truncate=False)

In [0]:
total_rows = pmt.count()

null_pct = (
    null_counts
    .toPandas()
    .T
    .reset_index()
)

null_pct.columns = ["column", "null_count"]

null_pct["null_pct"] = (
    null_pct["null_count"] / total_rows * 100
)

null_pct = null_pct[
    null_pct["null_count"] > 0
].sort_values("null_pct", ascending=False)

null_pct

In [0]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=null_pct,
    x="null_pct",
    y="column",
    hue="column",
    legend=False,
    palette="Reds",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Missing Values by Column",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Missing Values (%)")
ax.set_ylabel("Column")

ax.grid(axis="x", linestyle="--", alpha=0.7)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f%%",
        padding=3
    )

plt.tight_layout()
plt.show()

#### Missing Value & MNAR Analysis: DAYS_ENTRY_PAYMENT & AMT_PAYMENT

Both `DAYS_ENTRY_PAYMENT` and `AMT_PAYMENT` contain exactly **2,905 missing values** (0.0213%), while all scheduled/contractual columns contain zero missing values.

* **Identical Missingness Mechanism:** `DAYS_INSTALMENT` and `AMT_INSTALMENT` define the scheduled bill, whereas `DAYS_ENTRY_PAYMENT` and `AMT_PAYMENT` capture the execution of the transaction. If a customer fails to pay an installment, neither a payment date nor a payment amount can be logged, resulting in simultaneous nulls.
* **MNAR (Missing Not At Random):** The missingness is highly informative. Data is not missing at random; it is missing precisely because the repayment event did not occur (uncollected/defaulted installment).

---

### 2. 🚗 Point of Sale Cash Balance

The **`POS_CASH_balance`** table is a monthly snapshot history of past Point-of-Sale (POS) loans—such as store financing for electronics or appliances and personal cash loans issued by Home Credit.

Each row represents a single month's status update for a specific loan contract. It records the state of the account during that month, including whether the contract was active or completed, the number of installments remaining, and whether the payment for that month was on time or overdue.

In [0]:
print(f"({pos.count()}, {len(pos.columns)})")

#### Identifier Keys: SK_ID_CURR & SK_ID_PREV

`SK_ID_CURR` represents the unique identification number assigned to an individual customer, identifying who holds the credit history. `SK_ID_PREV` represents the unique identification number assigned to a specific past loan contract, distinguishing separate credit agreements held by that same customer.

Together, these two keys establish the relational structure of the table. A single customer (`SK_ID_CURR`) can have multiple past loan contracts (`SK_ID_PREV`). Because this table tracks month-by-month account balances, these IDs repeat across multiple rows—each row representing a single month's snapshot for a specific loan agreement.

---

In [0]:
pos.select("MONTHS_BALANCE").show(5)

#### `MONTHS_BALANCE`

A relative time index indicating the number of months prior to the current loan application date. 

Values are represented as non-positive integers, where the current application serves as the baseline:
* **`0`**: The month of the current application.
* **`-1`**: One month prior to the application.
* **`-12`**: Twelve months (one year) prior to the application.

In [0]:
pos.select("MONTHS_BALANCE").summary().show()

In [0]:
# histpplot
pos.select("MONTHS_BALANCE").plot.hist()

In [0]:
# box plot
pos.select("MONTHS_BALANCE").plot.box()

---

#### CNT_INSTALMENT

`CNT_INSTALMENT` specifies the total number of monthly installments agreed upon for a loan contract during that snapshot month. It represents the overall scheduled duration of the loan (for example, a value of 12 indicates a 12-month repayment plan).

While this value generally remains constant across all months of a loan, variations in `CNT_INSTALMENT` over time highlight contract modifications or term restructurings, such as a customer extending their repayment timeline to reduce their monthly bill.

In [0]:
pos.select("CNT_INSTALMENT").summary().show()

In [0]:
pos.select("CNT_INSTALMENT").orderBy("CNT_INSTALMENT", ascending=False).limit(10).show()

In [0]:
pos.select("CNT_INSTALMENT").plot.hist()

In [0]:
pos.groupBy("CNT_INSTALMENT").count().orderBy("count", ascending=False).limit(10).show()

#### Insights from CNT_INSTALMENT Distribution

The distribution of loan durations reflects standard banking products and retail financing agreements:

* **Standard Term Multiples:** Repayment plans heavily cluster around standard calendar intervals—primarily 12, 24, 6, 18, and 36 months—matching standard 6-month and 1-year credit products.
* **12-Month Dominance:** The 12-month tenure is the most common contract length, serving as the benchmark duration for Point-of-Sale financing (e.g., electronics and consumer goods).
* **Promotional Schemes:** Non-standard intervals like 10 and 8 months highlight promotional financing offers structured in collaboration with retail partners.
* **Short-Term Concentration:** The vast majority of contracts are short to medium term (>24 months), reflecting the low-ticket, quick-turnover nature of POS and consumer cash loans.

In [0]:
# box plot
pos.select("CNT_INSTALMENT").plot.box()

---

#### CNT_INSTALMENT_FUTURE

`CNT_INSTALMENT_FUTURE` represents the number of remaining monthly installments left on the loan contract at the time of the snapshot. Unlike `CNT_INSTALMENT` (which records the total contracted loan duration), this feature acts as a countdown timer, decreasing month-by-month as payments are completed until it reaches zero.

##### CNT_INSTALMENT == CNT_INSTALMENT_FUTURE

When `CNT_INSTALMENT` equals `CNT_INSTALMENT_FUTURE`, 100% of the loan installments remain unpaid. 

* **Baseline State:** This indicates the start of a new loan contract where no monthly payments have been processed yet.
* **Risk Signal (If persistent):** If this equality holds across several consecutive monthly snapshots, it signals a complete failure to initiate repayment, serving as an early indicator of first-payment default.

In [0]:
# Flag months where no payments have been made yet
df_flagged = pos.withColumn(
    "IS_UNSTARTED",
    sf.when(sf.col("CNT_INSTALMENT") == sf.col("CNT_INSTALMENT_FUTURE"), 1).otherwise(0)
)

# Aggregate by contract (SK_ID_PREV)
contract_analysis = df_flagged.groupBy("SK_ID_PREV", "SK_ID_CURR").agg(
    sf.sum("IS_UNSTARTED").alias("UNSTARTED_MONTHS_COUNT"),
    sf.count("MONTHS_BALANCE").alias("TOTAL_CONTRACT_MONTHS")
)

# Categorize into Normal Start vs Persistent Default
contract_cases = contract_analysis.withColumn(
    "START_STATUS",
    sf.when(sf.col("UNSTARTED_MONTHS_COUNT") == 1, "NORMAL_START")
     .when(sf.col("UNSTARTED_MONTHS_COUNT") > 1, "PERSISTENT_DEFAULT")
     .otherwise("STARTED_IMMEDIATELY")
)

# Show counts for each case
contract_cases.groupBy("START_STATUS").count().show()

In [0]:
status_df = (
    contract_cases
    .groupBy("START_STATUS")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 8))

wedges, _, _ = ax.pie(
    status_df["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    status_df["START_STATUS"],
    title="Start Status",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "Contract Start Status Distribution",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

---

#### NAME_CONTRACT_STATUS

`NAME_CONTRACT_STATUS` captures the operational state of a loan contract during a given monthly snapshot (`MONTHS_BALANCE`). It tracks how a loan progresses through its lifecycle over time.

* **`Active`**: The contract is open and active with scheduled monthly payments.
* **`Completed`**: The contract has been fully paid off and closed.
* **`Demand`**: A critical default state where the bank demands immediate payoff of the total balance due to severe non-payment.
* **`Signed` / `Approved`**: Early origination states before regular billing begins.
* **`Amortized debt` / `Returned to the store`**: Special administrative handling for bad debt restructurings or store returns.

In [0]:
pos.groupBy("NAME_CONTRACT_STATUS").count().show(truncate=-1)

In [0]:
status_df = (
    pos
    .groupBy("NAME_CONTRACT_STATUS")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=status_df,
    x="NAME_CONTRACT_STATUS",
    y="count",
    palette="pastel",
    edgecolor="black",
    linewidth=1.2,
    ax=ax,
    hue="NAME_CONTRACT_STATUS",
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=3,
        fontsize=10,
        fontweight="bold",
    )

ax.set_title(
    "Distribution of POS Contract Status",
    fontsize=16,
    fontweight="bold",
    pad=15,
)

ax.set_xlabel("Contract Status", fontsize=12)
ax.set_ylabel("Count", fontsize=12)

plt.xticks(rotation=20)
sns.despine()

plt.tight_layout()
plt.show()

---

#### SK_DPD (Days Past Due)

`SK_DPD` tracks how many days late a borrower was on their monthly payment for that specific snapshot month. Think of it as a late-payment ticker that starts counting the moment a due date passes without payment being received.

* **`SK_DPD = 0` (On Time):** The payment was made on or before the due date, or no overdue balance existed for that month.
* **`SK_DPD = 1 to 29` (Minor Delay):** The payment was slightly late. This often happens due to minor cash flow delays, technical glitches, or simple forgetfulness.
* **`SK_DPD = 30+` (Significant Delay):** The customer missed a full billing cycle. Banks view 30+ days overdue as a clear sign of financial distress.
* **`SK_DPD = 90+` (Severe Delinquency):** The customer is 3 or more months behind on payments. In consumer lending, hitting 90+ DPD is the standard threshold for considering a loan in default.

**Key Characteristic:** `SK_DPD` is a **monthly status**, not a permanent score. If a customer is 20 days late in Month -3 (`SK_DPD = 20`) but clears their dues before Month -2, their `SK_DPD` for Month -2 resets back to `0`.

#### SK_DPD_DEF (Days Past Due with Tolerance)

`SK_DPD_DEF` tracks overdue days specifically for significant unpaid balances that exceed the bank's default tolerance threshold. 

* **Difference from `SK_DPD`:** While `SK_DPD` logs any unpaid amount (even a few cents), `SK_DPD_DEF` ignores minor residual amounts (e.g., rounding differences or negligible fee leftovers).
* **`SK_DPD_DEF = 0`:** Indicates the loan balance is either fully settled or any remaining unpaid balance falls within acceptable tolerance limits.
* **`SK_DPD_DEF > 0`:** Indicates a material, true payment delinquency.

In [0]:
# box plot
pos.select("SK_DPD").plot.box()

In [0]:
# box plot
pos.select("SK_DPD_DEF").plot.box()

In [0]:
pos.agg(
    sf.mean("SK_DPD").alias("mean_SK_DPD"),
    sf.median("SK_DPD").alias("median_SK_DPD")
).show()

pos.agg(
    sf.mean("SK_DPD_DEF").alias("mean_SK_DPD_DEF"),
    sf.median("SK_DPD_DEF").alias("median_SK_DPD_DEF")
).show()

##### Zero vs. Non-Zero Frequency (Sparsity Check)

In [0]:
total_rows = pos.count()

sk_dpd_df = (
    pos
    .select(
        sf.when(sf.col("SK_DPD") == 0, "No Delay (0)")
        .when(sf.col("SK_DPD") > 0, "Delay (>0)")
        .otherwise("Missing")
        .alias("SK_DPD_STATUS")
    )
    .groupBy("SK_DPD_STATUS")
    .count()
    .withColumn(
        "percentage",
        sf.round(sf.col("count") / total_rows * 100, 2)
    )
)

sk_dpd_df.show(truncate=False)

# Pie Plot
sk_dpd_pd = sk_dpd_df.toPandas()

fig, ax = plt.subplots(figsize=(8, 8))

wedges, _, _ = ax.pie(
    sk_dpd_pd["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    sk_dpd_pd["SK_DPD_STATUS"],
    title="SK_DPD Status",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "SK_DPD Distribution",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [0]:
total_rows = pos.count()

sk_dpd_def_df = (
    pos
    .select(
        sf.when(sf.col("SK_DPD_DEF") == 0, "No Default Delay (0)")
        .when(sf.col("SK_DPD_DEF") > 0, "Default Delay (>0)")
        .otherwise("Missing")
        .alias("SK_DPD_DEF_STATUS")
    )
    .groupBy("SK_DPD_DEF_STATUS")
    .count()
    .withColumn(
        "percentage",
        sf.round(sf.col("count") / total_rows * 100, 2)
    )
)

sk_dpd_def_df.show(truncate=False)

# Pie Plot
sk_dpd_def_pd = sk_dpd_def_df.toPandas()

fig, ax = plt.subplots(figsize=(8, 8))

wedges, _, _ = ax.pie(
    sk_dpd_def_pd["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    sk_dpd_def_pd["SK_DPD_DEF_STATUS"],
    title="SK_DPD_DEF Status",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "SK_DPD_DEF Distribution",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [0]:
pos.show(5)

##### Non-Zero Percentile & Severity Distribution

In [0]:
sk_dpd_positive = (
    pos
    .filter(sf.col("SK_DPD") > 0)
)

quantiles = sk_dpd_positive.approxQuantile(
    "SK_DPD",
    [0.0, 0.50, 0.75, 0.90, 0.95, 1.0],
    0.01
)

print(f"Min: {quantiles[0]:.0f}")
print(f"Median: {quantiles[1]:.0f}")
print(f"75%: {quantiles[2]:.0f}")
print(f"90%: {quantiles[3]:.0f}")
print(f"95%: {quantiles[4]:.0f}")
print(f"Max: {quantiles[5]:.0f}")

In [0]:
# boxplot
sk_dpd_positive.select("SK_DPD").plot.box()

In [0]:
sk_dpd_def_positive = (
    pos
    .filter(sf.col("SK_DPD_DEF") > 0)
)

quantiles = sk_dpd_def_positive.approxQuantile(
    "SK_DPD_DEF",
    [0.0, 0.50, 0.75, 0.90, 0.95, 1.0],
    0.01
)

print(f"Min: {quantiles[0]:.0f}")
print(f"Median: {quantiles[1]:.0f}")
print(f"75%: {quantiles[2]:.0f}")
print(f"90%: {quantiles[3]:.0f}")
print(f"95%: {quantiles[4]:.0f}")
print(f"Max: {quantiles[5]:.0f}")

In [0]:
pos.filter(sf.col("SK_DPD_DEF") > 25).count()

##### SK_DPD & SK_DPD_DEF Distribution

The extreme values in **`SK_DPD`** (reaching up to 4,231 days, or nearly 11 years) are caused by "ghost" administrative balances rather than real customer default behavior. Because strict `SK_DPD` flags *any* unpaid amount even a few leftover cents from rounding or tiny bank fees the late-day counter keeps ticking indefinitely long after the customer has stopped using the account. This creates massive multi-year outliers that distort actual credit risk.

**`SK_DPD_DEF`** solves this by ignoring negligible balances, revealing that 95% of genuine customer payment delays are actually resolved within 25 days.

In [0]:
# boxplot
sk_dpd_def_positive.select("SK_DPD").plot.box()

#### Delinquency Severity — Nuisance vs. True Defaults

In [0]:
pos.filter(
    (sf.col("SK_DPD") > 0) &
    (sf.col("SK_DPD_DEF") == 0)
).count()

In [0]:
pos.filter(
    (sf.col("SK_DPD") > 0) &
    (sf.col("SK_DPD_DEF") > 0)
).count()

By comparing the raw Days Past Due (`SK_DPD`) against the bank's strict tolerance threshold (`SK_DPD_DEF`), we can cleanly divide historical late payments into two distinct behavioral profiles. 

**1. The "Administrative" Delays (181,258 records)**
*`Filter: (SK_DPD > 0) & (SK_DPD_DEF == 0)`*
This larger group represents borrowers who were technically late, but the bank did not penalize them or classify them as severely delinquent. In consumer lending, this usually indicates "nuisance" delays. These borrowers might have missed a tiny $2 administrative fee, paid just a few days late within an unwritten grace period, or experienced a bank transfer lag. From a risk perspective, this group is disorganized or forgetful, but they are not necessarily in financial distress. 

**2. The "Severe" Defaults (113,969 records)**
*`Filter: (SK_DPD > 0) & (SK_DPD_DEF > 0)`*
This smaller group represents true financial delinquency. Not only were these payments late, but they crossed Home Credit's internal threshold for severity (likely a missed principal payment or a heavily aged debt). These 113k records act as our strongest historical red flags. The borrower did not just forget a minor fee; they actively failed to meet a core financial obligation. 

---

### 3. 💳 Credit Card Balance
The `credit_card_balance` table is a monthly activity log for any credit cards a borrower previously held with Home Credit. Each row represents a single month's credit card statement for a specific card account. It records how the card was used during that month—including the overall credit limit, current balance, spent amounts, cash withdrawals, minimum payments due, actual amounts paid, and whether the monthly bill was paid on time or went overdue.

In [0]:
print(f"({cc.count()}, {len(cc.columns)})")

#### AMT_BALANCE

`AMT_BALANCE` represents the total outstanding debt owed on the credit card account at the end of a monthly billing cycle. It captures the exact balance remaining on the statement date, incorporating all accumulated purchases, cash advances, interest, and administrative fees.

In [0]:
cc.select("AMT_BALANCE").summary().show()

##### Key Takeaways from `AMT_BALANCE` Summary

1. Over 50% of Monthly Statements Have $0 Balance  
Both the 25th percentile and the 50th percentile (median) are 0.0. This means more than half of all logged monthly credit card statements reflect no outstanding debt either because the card was inactive, or the customer fully paid off their balance before the statement cutoff.

2. Heavy Right Skewness  
The mean (58,300) is much higher than the median (0.0), and the standard deviation (106,307) is nearly double the mean. This tells us that a smaller portion of active cardholders carries large balances, dragging the average upward:

* Bottom 50%: $0 balance

* 75th Percentile: ~$89,052 balance

* Max Value: ~$1,505,902 balance (an extreme maximum)

3. Massive Negative Outlier (-420,250.18)  
The minimum value shows a credit balance of over -$420,000. In real-world core banking systems, a negative balance this huge usually stems from a massive dispute settlement, loan reversal, or corporate refund.

In [0]:
# applicants with Credit Balance
cc.filter(sf.col("AMT_BALANCE") < 0).count()

In [0]:
cc.filter(sf.col("AMT_BALANCE") < 0).select("AMT_BALANCE").orderBy("AMT_BALANCE").show(10)

In [0]:
# zero outstanding debt / total rows
cc.filter(sf.col("AMT_BALANCE") == 0).count() / cc.count()

##### Negative Values in AMT_BALANCE (Credit Balances)

A negative AMT_BALANCE indicates a **Credit Balance**, meaning the customer has extra money credited to their account and the bank effectively owes that amount back to the borrower. 

In credit card accounting, a negative balance occurs whenever total credits applied during a billing cycle exceed total debits:

AMT_BALANCE = Beginning Balance + Debits (Purchases & Fees) - Credits (Payments & Refunds)

**Primary Causes:**
* **Overpayment:** A borrower pays more than their required bill (for example, paying $200 on a $150 balance leaves an AMT_BALANCE of -$50).
* **Merchant Returns:** A customer pays off their monthly bill in full and then returns a previously purchased item. The merchant refund posts as a credit to a zero-balance account.
* **Fee Reversals & Statement Credits:** The bank waives a disputed fee or issues promotional cashback rewards to an account that currently has a $0 balance.

---

####AMT_CREDIT_LIMIT_ACTUAL 
It is the approved maximum credit limit on the credit card for that specific monthly statement. It represents the total borrowing capacity granted to the customer by the bank during that month.

*Note: Credit card limits on a single card are actually dynamic and change over the life of the account. While a limit might stay constant for months or years at a time, banks actively adjust credit lines based on customer behavior and risk evaluations.*

In [0]:
cc.select("AMT_CREDIT_LIMIT_ACTUAL").plot.hist()

In [0]:
cc.select("AMT_CREDIT_LIMIT_ACTUAL").summary().show()

In [0]:
cc.filter(sf.col("AMT_CREDIT_LIMIT_ACTUAL") == 0).count()

In [0]:
# %age of Credit card with limit = 0
(cc.filter(sf.col("AMT_CREDIT_LIMIT_ACTUAL") == 0).count() / cc.count())*100

##### Wait Credit Limit Can Be 0?

There are three main reasons why a credit card limit shows as 0 in a monthly snapshot:

1. Frozen or Blocked Cards: If a borrower misses payments or enters severe delinquency, the bank reduces the active borrowing limit to 0 to block any new purchases or cash withdrawals while keeping the account open to collect remaining payments.

2. Closed Accounts with Remaining Debt: If an account is closed or canceled, the limit drops to 0, but the monthly statement continues to log if an outstanding balance (AMT_BALANCE) is still being paid off.

3. Unactivated or Restricted Accounts: Cards that are issued but not yet activated, or restricted during administrative review, will log a limit of 0.

----


#### AMT_DRAWINGS_ATM_CURRENT

This is the total amount of cash the customer took out of an ATM using their credit card during that month.

Taking cash out of an ATM with a credit card usually comes with high extra fees, so high values can show that a customer was running low on cash that month.

In [0]:
# summary stats
cc.select("AMT_DRAWINGS_ATM_CURRENT").summary().show()

In [0]:
# ratio of monthly credit card statements that include at least one ATM cash withdrawal (Cash Advance) in %age
(
    cc.filter(
        sf.col("AMT_DRAWINGS_ATM_CURRENT") > 0
    ).count()
    / cc.filter(
        sf.col("AMT_DRAWINGS_ATM_CURRENT").isNotNull()
    ).count()
) * 100

In [0]:
atm_withdrawal_df = (
    cc
    .filter(sf.col("AMT_DRAWINGS_ATM_CURRENT").isNotNull())
    .select(
        sf.when(
            sf.col("AMT_DRAWINGS_ATM_CURRENT") == 0,
            "No Withdrawal"
        )
        .when(
            sf.col("AMT_DRAWINGS_ATM_CURRENT") > 0,
            "Withdrawal"
        )
        .alias("ATM_WITHDRAWAL_STATUS")
    )
    .filter(sf.col("ATM_WITHDRAWAL_STATUS").isNotNull())
    .groupBy("ATM_WITHDRAWAL_STATUS")
    .count()
)

# convert to Pandas
atm_withdrawal_pd = atm_withdrawal_df.toPandas()

# plot
fig, ax = plt.subplots(figsize=(7, 7))

wedges, _, _ = ax.pie(
    atm_withdrawal_pd["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    atm_withdrawal_pd["ATM_WITHDRAWAL_STATUS"],
    title="ATM Withdrawal",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "ATM Withdrawal Distribution",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

In [0]:
# summary stats of AMT_DRAWINGS_ATM_CURRENT>0
cc.filter(sf.col("AMT_DRAWINGS_ATM_CURRENT") > 0).select("AMT_DRAWINGS_ATM_CURRENT").summary().show()

##### Negative Value in AMT_DRAWINGS_ATM_CURRENT!

In [0]:
# counting negative values
cc.filter(sf.col("AMT_DRAWINGS_ATM_CURRENT") < 0).count()

In [0]:
cc.filter(sf.col("AMT_DRAWINGS_ATM_CURRENT") < 0).show()

---

#### `AMT_DRAWINGS_CURRENT` (Total Monthly Spending)

The total monetary amount the borrower spent or withdrew using their credit card during that specific monthly billing cycle. This is the sum of all transaction types combined, including retail purchases, ATM cash advances, and any other miscellaneous charges.

In [0]:
cc.select("AMT_DRAWINGS_CURRENT").summary().show()

In [0]:
cc.filter(sf.col("AMT_DRAWINGS_CURRENT") < 0).count()

In [0]:
cc.filter(sf.col("AMT_DRAWINGS_CURRENT") < 0).show()

In [0]:
# box plot
cc.filter(sf.col("AMT_DRAWINGS_CURRENT") > 0).select("AMT_DRAWINGS_CURRENT").plot.box()

In [0]:
cc.filter(sf.col("AMT_DRAWINGS_CURRENT") > 0).select("AMT_DRAWINGS_CURRENT").plot.hist()

---

#### `AMT_DRAWINGS_POS_CURRENT`

The total monetary amount the borrower spent using their credit card at Point of Sale (POS) terminals during that specific monthly billing cycle. This represents normal retail purchases (e.g., swiping or tapping the card at a physical store, or making online checkouts), as opposed to ATM cash withdrawals.

In [0]:
cc.select("AMT_DRAWINGS_POS_CURRENT").summary().show()

In [0]:
cc.filter(sf.col("AMT_DRAWINGS_POS_CURRENT") > 0).select("AMT_DRAWINGS_POS_CURRENT").summary().show()

##### Ratio Between AMT_DRAWINGS_CURRENT / AMT_DRAWINGS_CURRENT

In [0]:
cc.select(
    "AMT_DRAWINGS_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    sf.when(
        sf.col("AMT_DRAWINGS_CURRENT") > 0, 
        sf.col("AMT_DRAWINGS_POS_CURRENT") / sf.col("AMT_DRAWINGS_CURRENT")
    ).otherwise(0).alias("POS_SPENDING_RATIO")
).select("POS_SPENDING_RATIO") \
.plot.hist(title="Histogram Plot of Ratio Between AMT_DRAWINGS_POS_CURRENT and AMT_DRAWINGS_CURRENT")

##### The Binary Nature of Credit Card Usage

The ratio of Point-of-Sale (POS) spending to total credit card spending reveals a stark, extreme bimodal distribution. Borrowers overwhelmingly fall into one of two polarized groups, with almost no one mixing these behaviors in a single billing cycle.

**Group 1: The Cash-Seekers (Ratio near 0.0)**
The massive spike in the `[0.0, 0.1)` bin (containing roughly 3.5 million records) represents months where POS spending made up 0% (or near 0%) of the total amount drawn. This means these borrowers are actively using their credit lines, but *not* for retail shopping. Their total spending is being driven entirely by other transaction types, which in the context of distressed credit, heavily implies ATM cash advances. 

**Group 2: The Pure Shoppers (Ratio near 1.0)**
The secondary, smaller peak in the `[0.9, 1.0]` bin represents months where POS spending accounted for nearly 100% of all credit card activity. These borrowers are using their credit cards exactly as intended: strictly for normal retail purchases at physical or digital storefronts, without resorting to cash withdrawals. 

* Ratio = 1.0 (100%): The borrower is using the card purely for regular store purchases. This is standard, low-risk consumer behavior.

* Ratio = 0.0 (0%): If the total spending is high, but the POS ratio is zero, it means 100% of their spending was withdrawn as hard cash from an ATM. This flags severe financial desperation.

##### Average Monetary Amount Spent at Point of Sale (POS) per Customer

In [0]:
(
    cc
    .groupBy("SK_ID_CURR")
    .agg(
        sf.mean("AMT_DRAWINGS_POS_CURRENT").alias(
            "AVG_AMT_DRAWINGS_POS_CURRENT"
        )
    )
    .select("AVG_AMT_DRAWINGS_POS_CURRENT")
).summary().show()

---

#### `AMT_INST_MIN_REGULARITY` (Minimum Debt Repayment Required)

This is the **absolute minimum amount of money the borrower must pay back to the bank** for that specific month to avoid a late penalty. 

Despite the confusing word "regularity," this feature has **nothing to do with spending money or keeping the card active**. It is strictly about repaying the debt the borrower already owes. 

**How it works in the real world:**
* If a borrower uses their card and owes the bank $1,000, the bank will say: *"You don't have to pay all $1,000 back today, but you MUST pay us at least **$35** this month."* That $35 is recorded in this column. 
* If a borrower hasn't used their card and owes $0, this minimum required payment is simply $0. 

In [0]:
cc.select("AMT_INST_MIN_REGULARITY").summary().show()

In [0]:
cc.select(
    sf.mean(sf.when(sf.col("AMT_INST_MIN_REGULARITY") == 0, 1.0).otherwise(0.0)).alias("PCT_ZERO"),
    sf.mean(sf.when(sf.col("AMT_INST_MIN_REGULARITY") > 0, 1.0).otherwise(0.0)).alias("PCT_POSITIVE"),
    sf.mean(sf.when(sf.col("AMT_INST_MIN_REGULARITY").isNull(), 1.0).otherwise(0.0)).alias("PCT_NULL")
).show()

In [0]:
(
    cc
    .select(
        sf.log1p("AMT_INST_MIN_REGULARITY")
        .alias("LOG_AMT_INST_MIN_REGULARITY")
    )
    .plot
    .kde(
        1, title="KDE Plot of Log(AMT_INST_MIN_REGULARITY)"
    )
)

* The sharp mode at $0$ corresponds to billing cycles with `AMT_INST_MIN_REGULARITY` = 0. In these months, the borrower carried no revolving balance, incurred no fees, and had no minimum payment obligation to the bank.

* The second broad distribution captures active, debt-carrying statement cycles

#### `AMT_PAYMENT_CURRENT`

The actual monetary amount the borrower paid toward their credit card balance during that specific monthly billing cycle. 

Unlike `AMT_INST_MIN_REGULARITY` (which is what the bank demanded), this column records what the customer **actually transferred or deposited** to pay down their debt that month.

---

#### `AMT_PAYMENT_TOTAL_CURRENT`

The grand total of all monetary amounts paid by the applicant on the credit card account during that monthly cycle, encompassing both their standard balance payments and any additional secondary settlements (such as separate fee clearings, penalty settlements, or cross-account adjustments).

---

#### Null Values Analysis

In [0]:
# null values
null_counts = cc.select([
    sf.count(sf.when(sf.col(c).isNull(), c)).alias(c)
    for c in cc.columns
])
null_counts.show(truncate=False)

total_rows = cc.count()

null_pct = (
    null_counts
    .toPandas()
    .T
    .reset_index()
)

null_pct.columns = ["column", "null_count"]

null_pct["null_pct"] = (
    null_pct["null_count"] / total_rows * 100
)

null_pct = null_pct[
    null_pct["null_count"] > 0
].sort_values("null_pct", ascending=False)

null_pct

##### Interpretation of Missing Data (NULL Mechanics)

Missing values in this table are structural artifacts of transactional event logging rather than random data loss. When an action does not occur during a statement cycle, the system omits the entry instead of recording an explicit zero.

**1. Channel Drawings & Counts (19.52% NULL)**
* `AMT_DRAWINGS_ATM_CURRENT`, `AMT_DRAWINGS_POS_CURRENT`, `AMT_DRAWINGS_OTHER_CURRENT`
* `CNT_DRAWINGS_ATM_CURRENT`, `CNT_DRAWINGS_POS_CURRENT`, `CNT_DRAWINGS_OTHER_CURRENT`

* **Why it is missing:** The core banking engine only generates drawing records when an actual transaction clears through a specific payment channel. When an account is dormant or a specific channel is unused during that monthly cycle, the ledger leaves the sub-channel entry empty.
* **What it means:** The borrower had **0 transactions** and spent **$0.00** through that specific channel during that billing cycle.

**2. Actual Repayment Amount (20.00% NULL)**
* `AMT_PAYMENT_CURRENT`

* **Why it is missing:** This column tracks inbound payment logs. Missingness occurs when either the account carried no balance (no payment requested) or the borrower carried a balance but failed to make any payment during that monthly cycle.
* **What it means:** The customer deposited/transferred **$0.00** toward their revolving credit balance for that month.

**3. Minimum Payment Due & Cumulative Installments (7.95% NULL)**
* `AMT_INST_MIN_REGULARITY`
* `CNT_INSTALMENT_MATURE_CUM`

* **Why it is missing:** The installment schedule generator only initializes once the first billing cycle officially closes on an active balance. Accounts in their initial opening grace period or unused cards with a continuous $0.0 balance have not generated a matured billing cycle.
* **What it means:** The customer has **0 matured installment cycles** and a **$0.00 minimum payment obligation** to date.